# Fase III — Limpieza de datos
## Proyecto de cátedra · Data Analysis · Ames Housing

**Dataset:** Ames Housing (De Cock, 2011) — 2,930 observaciones × 82 variables
**Entrada:** `data/raw/AmesHousing.csv` (inmutable)
**Salida:** `data/processed/AmesHousing.csv` (modificable)

### Alcance de esta fase
| Incluido | Excluido (corresponde a Fase IV) |
|---|---|
| Tratamiento de valores faltantes | Escalamiento (Min-Max / Standard / Robust) |
| Detección y tratamiento de outliers | Encoding de categóricas |
| Estandarización de texto y fechas | Creación de variables derivadas |
| Duplicados exactos y parciales | Selección de variables |
| Corrección de errores de captura | |

> **Regla operativa:** este notebook lee de `raw/` y escribe en `processed/`.
> El archivo crudo nunca se modifica. Todo resultado es reproducible ejecutando
> el notebook completo de principio a fin.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

# Resolución de rutas: funciona tanto desde notebooks/ como desde la raíz del repo
BASE = Path.cwd()
ROOT = BASE if (BASE / "data").exists() else BASE.parent

RAW       = ROOT / "data" / "raw" / "AmesHousing.csv"
PROCESSED = ROOT / "data" / "processed" / "AmesHousing.csv"
DOCS      = ROOT / "docs"
DOCS.mkdir(exist_ok=True)

# Etiqueta para codificar la ausencia de un atributo.
# NO se usa "None": pandas la interpreta como valor faltante al leer el CSV,
# lo que destruiria la categoria al exportar (verificado en la seccion 11).
ETIQUETA_AUSENCIA = "Ninguno"

df_raw = pd.read_csv(RAW)
df = df_raw.copy()

print(f"Origen : {RAW}")
print(f"Destino: {PROCESSED}")
print(f"Forma inicial: {df.shape[0]:,} filas × {df.shape[1]} columnas")

Origen : c:\Users\Jhonny\source\repos\analisis-datos-ames-housing\data\raw\AmesHousing.csv
Destino: c:\Users\Jhonny\source\repos\analisis-datos-ames-housing\data\processed\AmesHousing.csv
Forma inicial: 2,930 filas × 82 columnas


---
## 1. Snapshot "antes"

Las métricas del crudo se calculan **antes de tocar nada**. Una vez imputados los
nulos, la media y la desviación originales son irrecuperables. Estas cifras
alimentan la comparativa pre/post de la Fase V.

In [2]:
def resumen_numerico(data, etapa):
    """Estadísticos descriptivos de todas las variables numéricas."""
    num = data.select_dtypes(include="number")
    r = pd.DataFrame({
        "etapa":     etapa,
        "n_validos": num.notna().sum(),
        "nulos":     num.isna().sum(),
        "media":     num.mean(),
        "mediana":   num.median(),
        "desv_std":  num.std(),
        "minimo":    num.min(),
        "maximo":    num.max(),
        "skewness":  num.skew(),
        "curtosis":  num.kurt(),
    })
    return r.reset_index().rename(columns={"index": "variable"})


def contadores(data, etapa):
    """Indicadores globales de calidad del dataset."""
    no_num = data.select_dtypes(exclude="number")
    return {
        "etapa":              etapa,
        "filas":              len(data),
        "columnas":           data.shape[1],
        "celdas_nulas":       int(data.isna().sum().sum()),
        "pct_celdas_nulas":   round(data.isna().sum().sum() / data.size * 100, 3),
        "columnas_con_nulos": int((data.isna().sum() > 0).sum()),
        "duplicados_exactos": int(data.duplicated().sum()),
        "variables_numericas":     data.select_dtypes(include="number").shape[1],
        "variables_no_numericas":  no_num.shape[1],
    }


metricas_antes = resumen_numerico(df, "antes")
conteo_antes   = contadores(df, "antes")

pd.Series(conteo_antes)

etapa                     antes
filas                      2930
columnas                     82
celdas_nulas              15749
pct_celdas_nulas          6.555
columnas_con_nulos           27
duplicados_exactos            0
variables_numericas          39
variables_no_numericas       43
dtype: object

---
## 2. Diagnóstico de valores faltantes

El punto crítico de este dataset: **la mayoría de los `NaN` no son datos perdidos.**
En la documentación de De Cock, un `NaN` en `Pool QC` significa *la casa no tiene
piscina*, no *se desconoce la calidad de la piscina*. Imputar la moda ahí sería
inventar piscinas.

Por eso el tratamiento se divide en dos categorías, y esa distinción es el
argumento metodológico central de esta fase.

In [3]:
nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "pct":   (df.isna().sum() / len(df) * 100).round(2),
    "dtype": df.dtypes.astype(str),
})
nulos = nulos[nulos["nulos"] > 0].sort_values("nulos", ascending=False)

print(f"Columnas con al menos un nulo: {len(nulos)} de {df.shape[1]}")
print(f"Celdas nulas totales: {df.isna().sum().sum():,}")
nulos

Columnas con al menos un nulo: 27 de 82
Celdas nulas totales: 15,749


,nulos,pct,dtype
Pool QC,2917,99.56,str
Misc Feature,2824,96.38,str
Alley,2732,93.24,str
Fence,2358,80.48,str
Mas Vnr Type,1775,60.58,str
Fireplace Qu,1422,48.53,str
Lot Frontage,490,16.72,float64
Garage Qual,159,5.43,str
Garage Cond,159,5.43,str
Garage Yr Blt,159,5.43,float64


### 2.1 Verificación de la hipótesis estructural

No basta con afirmar que los nulos son estructurales: hay que **demostrarlo** con
una variable testigo. Si el `NaN` en `Fireplace Qu` significa "no tiene chimenea",
entonces todas esas filas deben tener `Fireplaces == 0`.

In [4]:
pruebas = [
    ("Pool QC",       "Pool Area",     "Pool Area == 0"),
    ("Fireplace Qu",  "Fireplaces",    "Fireplaces == 0"),
    ("Garage Type",   "Garage Area",   "Garage Area == 0"),
    ("Bsmt Qual",     "Total Bsmt SF", "Total Bsmt SF == 0"),
]

filas = []
for col, testigo, desc in pruebas:
    nulos_col = df[col].isna()
    coinciden = (nulos_col & (df[testigo].fillna(0) == 0)).sum()
    filas.append({
        "columna": col, "nulos": int(nulos_col.sum()),
        "testigo": desc, "confirmados": int(coinciden),
        "coincidencia_%": round(coinciden / nulos_col.sum() * 100, 2),
    })

verificacion = pd.DataFrame(filas)
display(verificacion)
print("\nCoincidencia del 100% => los nulos son estructurales (ausencia del atributo),")
print("no datos perdidos. Se codifican como categoria explicita, no se imputan.")

,columna,nulos,testigo,confirmados,coincidencia_%
0,Pool QC,2917,Pool Area == 0,2917,100.0
1,Fireplace Qu,1422,Fireplaces == 0,1422,100.0
2,Garage Type,157,Garage Area == 0,157,100.0
3,Bsmt Qual,80,Total Bsmt SF == 0,80,100.0



Coincidencia del 100% => los nulos son estructurales (ausencia del atributo),
no datos perdidos. Se codifican como categoria explicita, no se imputan.


---
## 3. Tratamiento de valores faltantes

### 3.1 Nulos estructurales en variables categóricas → `"Ninguno"`

> **Por qué no `"None"`:** `pandas.read_csv()` incluye la cadena `None` en su lista
> de valores nulos por defecto. Si se exportara con esa etiqueta, la categoría
> volvería a leerse como `NaN` y el trabajo de esta fase se perdería en silencio al
> abrir el archivo en la Fase IV. Se usa `"Ninguno"`, que no colisiona con ningún
> token de valor faltante.

In [5]:
CAT_ESTRUCTURALES = [
    "Pool QC", "Misc Feature", "Alley", "Fence", "Fireplace Qu",
    "Garage Type", "Garage Finish", "Garage Qual", "Garage Cond",
    "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure", "BsmtFin Type 1", "BsmtFin Type 2",
]

registro = []   # bitácora de operaciones

# Caso especial 1: la fila 2237 declara 'Detchd' pero no tiene superficie ni
# capacidad de garaje. Es una inconsistencia interna, no un garaje real.
mask_garaje_fantasma = (
    df["Garage Type"].notna() & df["Garage Area"].isna() & df["Garage Cars"].isna()
)
n_fantasma = int(mask_garaje_fantasma.sum())
df.loc[mask_garaje_fantasma, "Garage Type"] = np.nan
registro.append({
    "bloque": "Nulos", "variable": "Garage Type",
    "problema": "Tipo de garaje declarado sin superficie ni capacidad asociada",
    "metodo": "Reclasificado como ausencia de garaje",
    "filas_afectadas": n_fantasma,
    "justificacion": "Inconsistencia interna: un garaje sin area ni capacidad no existe fisicamente",
})

for col in CAT_ESTRUCTURALES:
    n = int(df[col].isna().sum())
    df[col] = df[col].fillna(ETIQUETA_AUSENCIA)
    registro.append({
        "bloque": "Nulos", "variable": col,
        "problema": "NaN por ausencia del atributo (no dato perdido)",
        "metodo": f'Codificado como categoria "{ETIQUETA_AUSENCIA}"',
        "filas_afectadas": n,
        "justificacion": "Verificado contra variable testigo: coincidencia 100%",
    })

print(f"Garajes fantasma reclasificados: {n_fantasma}")
print(f"Columnas categoricas tratadas: {len(CAT_ESTRUCTURALES)}")
print(f"Nulos restantes: {df.isna().sum().sum():,}")

Garajes fantasma reclasificados: 1
Columnas categoricas tratadas: 14
Nulos restantes: 2,458


### 3.2 Nulos estructurales en variables numéricas → `0`

Si la casa no tiene sótano, su superficie de sótano no es "desconocida": es cero.
Imputar la mediana aquí introduciría sótanos inexistentes en el 2.7% de la muestra.

In [6]:
NUM_ESTRUCTURALES = [
    "Garage Area", "Garage Cars",
    "Total Bsmt SF", "Bsmt Unf SF", "BsmtFin SF 1", "BsmtFin SF 2",
    "Bsmt Full Bath", "Bsmt Half Bath",
]

for col in NUM_ESTRUCTURALES:
    n = int(df[col].isna().sum())
    if n:
        df[col] = df[col].fillna(0)
        registro.append({
            "bloque": "Nulos", "variable": col,
            "problema": "NaN por ausencia de la estructura fisica",
            "metodo": "Imputado con 0",
            "filas_afectadas": n,
            "justificacion": "Ausencia de sotano/garaje implica superficie nula, no valor desconocido",
        })

# Mas Vnr: sin revestimiento => tipo "None" y area 0.
# Se distingue de los casos con area > 0, que sí son datos perdidos reales.
sin_revest = df["Mas Vnr Type"].isna() & (df["Mas Vnr Area"].fillna(0) == 0)
n_sin = int(sin_revest.sum())
df.loc[sin_revest, "Mas Vnr Type"] = ETIQUETA_AUSENCIA
df.loc[sin_revest, "Mas Vnr Area"] = 0
registro.append({
    "bloque": "Nulos", "variable": "Mas Vnr Type / Mas Vnr Area",
    "problema": "NaN por ausencia de revestimiento de mamposteria",
    "metodo": f'Tipo = "{ETIQUETA_AUSENCIA}", area = 0',
    "filas_afectadas": n_sin,
    "justificacion": "Area nula o cero confirma ausencia del atributo",
})

print(f"Viviendas sin revestimiento: {n_sin}")
print(f"Nulos restantes: {df.isna().sum().sum():,}")

Viviendas sin revestimiento: 1768
Nulos restantes: 657


### 3.3 Nulos reales → imputación justificada

Lo que queda son datos genuinamente perdidos. Cada uno recibe un método distinto
según su naturaleza, y cada método se justifica.

| Variable | Nulos | Método | Razón |
|---|---|---|---|
| `Lot Frontage` | 490 | Mediana **por barrio** | El frente de lote depende del trazado urbano; la mediana global ignora esa estructura |
| `Garage Yr Blt` | ~159 | `Year Built` de la vivienda | El garaje se construye con la casa salvo remodelación |
| `Mas Vnr Type` | 7 | Moda | Área > 0 confirma que hay revestimiento; solo falta el tipo |
| `Electrical` | 1 | Moda | Un único caso; la moda es 91% del total |
| `Garage Finish/Qual/Cond` | 1 | Moda del mismo `Garage Type` | Garaje real (360 sqft) con atributos sin registrar |

In [7]:
# Lot Frontage: mediana por barrio, con respaldo en la mediana global
n_lf = int(df["Lot Frontage"].isna().sum())
mediana_barrio = df.groupby("Neighborhood")["Lot Frontage"].transform("median")
df["Lot Frontage"] = df["Lot Frontage"].fillna(mediana_barrio)
df["Lot Frontage"] = df["Lot Frontage"].fillna(df["Lot Frontage"].median())
registro.append({
    "bloque": "Nulos", "variable": "Lot Frontage",
    "problema": "Dato perdido real (16.72% de la muestra)",
    "metodo": "Mediana por Neighborhood (respaldo: mediana global)",
    "filas_afectadas": n_lf,
    "justificacion": "El frente de lote esta determinado por el trazado del barrio; la mediana global sobreestima barrios densos",
})

# Garage Yr Blt: año de construcción de la vivienda
n_gy = int(df["Garage Yr Blt"].isna().sum())
df["Garage Yr Blt"] = df["Garage Yr Blt"].fillna(df["Year Built"])
registro.append({
    "bloque": "Nulos", "variable": "Garage Yr Blt",
    "problema": "Sin garaje o ano no registrado",
    "metodo": "Imputado con Year Built",
    "filas_afectadas": n_gy,
    "justificacion": "El garaje se construye junto a la vivienda salvo remodelacion; evita introducir el valor 0 como ano",
})

# Detalles de garaje faltantes en un garaje que sí existe: moda dentro del mismo tipo
for col in ["Garage Finish", "Garage Qual", "Garage Cond"]:
    faltan = df[col].isna()
    if faltan.any():
        n = int(faltan.sum())
        for tipo in df.loc[faltan, "Garage Type"].unique():
            m = faltan & (df["Garage Type"] == tipo)
            moda = df.loc[(df["Garage Type"] == tipo) & df[col].notna(), col].mode()
            df.loc[m, col] = moda[0] if len(moda) else ETIQUETA_AUSENCIA
        registro.append({
            "bloque": "Nulos", "variable": col,
            "problema": "Garaje existente con atributo sin registrar",
            "metodo": "Moda condicionada al mismo Garage Type",
            "filas_afectadas": n,
            "justificacion": "Condicionar por tipo preserva la relacion entre tipo de garaje y su acabado",
        })

# Mas Vnr Type con área > 0: dato perdido real
faltan = df["Mas Vnr Type"].isna()
if faltan.any():
    n = int(faltan.sum())
    df.loc[faltan, "Mas Vnr Type"] = df["Mas Vnr Type"].mode()[0]
    registro.append({
        "bloque": "Nulos", "variable": "Mas Vnr Type",
        "problema": "Area de revestimiento > 0 pero tipo no registrado",
        "metodo": "Imputado con la moda",
        "filas_afectadas": n,
        "justificacion": "El area confirma la existencia del revestimiento; solo falta la clasificacion",
    })

# Mas Vnr Area residual
if df["Mas Vnr Area"].isna().any():
    n = int(df["Mas Vnr Area"].isna().sum())
    df["Mas Vnr Area"] = df["Mas Vnr Area"].fillna(0)
    registro.append({"bloque": "Nulos", "variable": "Mas Vnr Area",
                     "problema": "Area no registrada", "metodo": "Imputado con 0",
                     "filas_afectadas": n, "justificacion": "Ausencia de registro compatible con ausencia de revestimiento"})

# Electrical: caso único
if df["Electrical"].isna().any():
    n = int(df["Electrical"].isna().sum())
    moda = df["Electrical"].mode()[0]
    df["Electrical"] = df["Electrical"].fillna(moda)
    registro.append({
        "bloque": "Nulos", "variable": "Electrical",
        "problema": "Dato perdido puntual", "metodo": f"Imputado con la moda ({moda})",
        "filas_afectadas": n,
        "justificacion": "Un solo caso (0.03%); la moda concentra mas del 90% de las observaciones",
    })

print(f"Nulos restantes en el dataset: {df.isna().sum().sum()}")
assert df.isna().sum().sum() == 0, "Quedan nulos sin tratar"
print("OK — no quedan valores faltantes.")

Nulos restantes en el dataset: 0
OK — no quedan valores faltantes.


---
## 4. Corrección de errores de captura

Un valor puede no ser nulo y aun así ser imposible. `Garage Yr Blt = 2207` es un
error de digitación documentado en este dataset: la vivienda se construyó en 2006
y se vendió en 2007.

In [8]:
LIMITE = int(df["Yr Sold"].max())

imposibles = df.loc[df["Garage Yr Blt"] > LIMITE, ["Order", "Garage Yr Blt", "Year Built", "Yr Sold"]]
display(imposibles)

n_imp = len(imposibles)
if n_imp:
    # 2207 -> 2007: transposición de dígitos, coherente con el año de venta
    df.loc[df["Garage Yr Blt"] > LIMITE, "Garage Yr Blt"] = (
        df.loc[df["Garage Yr Blt"] > LIMITE, "Yr Sold"]
    )
    registro.append({
        "bloque": "Errores de captura", "variable": "Garage Yr Blt",
        "problema": f"Ano posterior al ultimo ano de venta registrado ({LIMITE})",
        "metodo": "Corregido al ano de venta de la vivienda",
        "filas_afectadas": n_imp,
        "justificacion": "Error de digitacion (2207 por 2007); la vivienda se construyo en 2006",
    })

# Validación de rangos: ninguna superficie ni precio puede ser negativo
negativos = {}
for col in df.select_dtypes(include="number").columns:
    if any(k in col for k in ["SF", "Area", "Price", "Val", "Porch"]):
        neg = int((df[col] < 0).sum())
        if neg:
            negativos[col] = neg

print(f"\nAnos imposibles corregidos: {n_imp}")
print(f"Variables con valores negativos invalidos: {len(negativos)} {negativos if negativos else ''}")
print(f"Rango Garage Yr Blt: {int(df['Garage Yr Blt'].min())} - {int(df['Garage Yr Blt'].max())}")

,Order,Garage Yr Blt,Year Built,Yr Sold
2260,2261,2207.0,2006,2007



Anos imposibles corregidos: 1
Variables con valores negativos invalidos: 0 
Rango Garage Yr Blt: 1872 - 2010


---
## 5. Estandarización de texto

Un espacio final invisible convierte `"WD "` y `"WD"` en dos categorías distintas.
Si esto no se corrige antes de la Fase IV, el One-Hot Encoding genera columnas
duplicadas.

In [9]:
col_texto = df.select_dtypes(exclude="number").columns

antes_categorias = {c: df[c].nunique() for c in col_texto}
afectadas = []

for col in col_texto:
    s = df[col].astype(str)
    if (s != s.str.strip()).any():
        afectadas.append((col, int((s != s.str.strip()).sum())))
    df[col] = s.str.strip()

for col, n in afectadas:
    registro.append({
        "bloque": "Estandarizacion", "variable": col,
        "problema": "Espacios en blanco al inicio o final del valor",
        "metodo": "Aplicado strip()",
        "filas_afectadas": n,
        "justificacion": 'Evita que "WD " y "WD" se traten como categorias distintas en el encoding',
    })

cambios_cardinalidad = {
    c: (antes_categorias[c], df[c].nunique())
    for c in col_texto if antes_categorias[c] != df[c].nunique()
}

print(f"Columnas de texto revisadas: {len(col_texto)}")
print(f"Columnas con espacios sobrantes: {len(afectadas)} -> {afectadas}")
print(f"Cambios en cardinalidad: {cambios_cardinalidad if cambios_cardinalidad else 'ninguno'}")

# Fechas: el dataset codifica la venta en Mo Sold / Yr Sold por separado.
# Se verifica su validez sin crear variables nuevas (eso es Fase IV).
print(f"\nMo Sold rango: {df['Mo Sold'].min()}-{df['Mo Sold'].max()} (valido: 1-12)")
print(f"Yr Sold rango: {df['Yr Sold'].min()}-{df['Yr Sold'].max()}")
assert df["Mo Sold"].between(1, 12).all(), "Meses fuera de rango"

Columnas de texto revisadas: 43
Columnas con espacios sobrantes: 1 -> [('Sale Type', 2536)]
Cambios en cardinalidad: ninguno

Mo Sold rango: 1-12 (valido: 1-12)
Yr Sold rango: 2006-2010


---
## 6. Duplicados

Se evalúan en tres niveles, porque `Order` y `PID` son identificadores únicos y
harían imposible detectar un registro repetido con distinto folio.

In [10]:
IDENTIFICADORES = ["Order", "PID"]

dup_exactos  = int(df.duplicated().sum())
dup_pid      = int(df.duplicated(subset=["PID"]).sum())
dup_parcial  = int(df.drop(columns=IDENTIFICADORES).duplicated().sum())

resumen_dup = pd.DataFrame([
    {"nivel": "Fila completa (exactos)",       "criterio": "las 82 columnas",         "duplicados": dup_exactos},
    {"nivel": "Identificador de propiedad",    "criterio": "PID",                     "duplicados": dup_pid},
    {"nivel": "Parcial (sin identificadores)", "criterio": "80 columnas sin Order/PID","duplicados": dup_parcial},
])
display(resumen_dup)

n_prev = len(df)
if dup_parcial:
    df = df.drop_duplicates(subset=[c for c in df.columns if c not in IDENTIFICADORES]).reset_index(drop=True)

registro.append({
    "bloque": "Duplicados", "variable": "dataset completo",
    "problema": "Verificacion en tres niveles (exacto, por PID, parcial sin identificadores)",
    "metodo": "drop_duplicates sobre las 80 columnas descriptivas" if dup_parcial else "Sin accion requerida",
    "filas_afectadas": n_prev - len(df),
    "justificacion": "Order y PID son unicos por construccion; evaluar solo filas completas ocultaria repeticiones reales",
})

print(f"Filas eliminadas por duplicidad: {n_prev - len(df)}")

,nivel,criterio,duplicados
0,Fila completa (exactos),las 82 columnas,0
1,Identificador de propiedad,PID,0
2,Parcial (sin identificadores),80 columnas sin Order/PID,0


Filas eliminadas por duplicidad: 0


---
## 7. Outliers

Se aplican **ambos** criterios que pide la guía. No coinciden, y esa discrepancia
es en sí misma un resultado: el IQR detecta más porque no asume normalidad, mientras
que el Z-score subestima en distribuciones asimétricas — precisamente porque la
media y la desviación ya están contaminadas por los valores extremos.

In [11]:
def detectar_outliers(serie):
    s = serie.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_iqr = int(((s < lim_inf) | (s > lim_sup)).sum())
    z = ((s - s.mean()) / s.std()).abs()
    return {
        "n_iqr": n_iqr, "pct_iqr": round(n_iqr / len(s) * 100, 2),
        "n_zscore": int((z > 3).sum()), "pct_zscore": round((z > 3).sum() / len(s) * 100, 2),
        "lim_inf": round(lim_inf, 2), "lim_sup": round(lim_sup, 2),
        "skewness": round(s.skew(), 3), "curtosis": round(s.kurt(), 3),
    }

CONTINUAS = [
    "SalePrice", "Gr Liv Area", "Lot Area", "Lot Frontage", "Total Bsmt SF",
    "1st Flr SF", "2nd Flr SF", "Garage Area", "Mas Vnr Area",
    "Wood Deck SF", "Open Porch SF", "Misc Val", "Pool Area",
]

reporte_outliers = pd.DataFrame(
    {c: detectar_outliers(df[c]) for c in CONTINUAS}
).T.reset_index().rename(columns={"index": "variable"})

display(reporte_outliers)
print("La divergencia IQR vs Z-score crece con la asimetria: en Lot Area y Misc Val,")
print("donde skewness > 10, el Z-score identifica una fraccion minima de los casos.")

,variable,n_iqr,pct_iqr,n_zscore,pct_zscore,lim_inf,lim_sup,skewness,curtosis
0,SalePrice,137.0,4.68,45.0,1.54,3500.00,339500.00,1.744,5.119
1,Gr Liv Area,75.0,2.56,25.0,0.85,200.88,2667.88,1.274,4.138
2,Lot Area,127.0,4.33,29.0,0.99,1267.75,17727.75,12.821,265.024
3,Lot Frontage,206.0,7.03,26.0,0.89,30.00,110.00,1.501,12.869
4,Total Bsmt SF,124.0,4.23,21.0,0.72,30.25,2064.25,1.151,9.107
5,1st Flr SF,43.0,1.47,29.0,0.99,114.62,2145.62,1.469,6.969
6,2nd Flr SF,8.0,0.27,10.0,0.34,-1055.62,1759.38,0.866,-0.415
7,Garage Area,42.0,1.43,17.0,0.58,-64.00,960.00,0.240,0.948
8,Mas Vnr Area,203.0,6.93,63.0,2.15,-244.12,406.88,2.619,9.369
9,Wood Deck SF,67.0,2.29,42.0,1.43,-252.00,420.00,1.843,6.754


La divergencia IQR vs Z-score crece con la asimetria: en Lot Area y Misc Val,
donde skewness > 10, el Z-score identifica una fraccion minima de los casos.


### 7.1 Decisión de tratamiento

**No se eliminan los outliers detectados estadísticamente.** Una casa de 12 acres
en Ames existe; no es un error. Eliminar por criterio puramente estadístico
destruiría el 4.7% de la muestra y sesgaría el análisis hacia la vivienda promedio.

La única remoción tiene fundamento **de dominio, no estadístico**: De Cock (2011)
documenta cinco viviendas de más de 4,000 pies² que distorsionan cualquier modelo.
Tres corresponden a ventas `Partial` — inmuebles no terminados al momento de la
venta, cuyo precio no refleja el valor de mercado del bien completo.

> Ver: De Cock, D. (2011). *Ames, Iowa: Alternative to the Boston Housing Data as an
> End of Semester Regression Project*. Journal of Statistics Education, 19(3).

In [12]:
UMBRAL_SUPERFICIE = 4000

candidatos = df.loc[
    df["Gr Liv Area"] > UMBRAL_SUPERFICIE,
    ["Order", "Gr Liv Area", "SalePrice", "Sale Condition", "Overall Qual"]
].copy()
candidatos["precio_por_sqft"] = (candidatos["SalePrice"] / candidatos["Gr Liv Area"]).round(2)
display(candidatos)

print(f"Precio medio por sqft del resto del dataset: "
      f"{(df['SalePrice'] / df['Gr Liv Area']).median():.2f}")

n_prev = len(df)
df = df[df["Gr Liv Area"] <= UMBRAL_SUPERFICIE].reset_index(drop=True)
eliminadas = n_prev - len(df)

registro.append({
    "bloque": "Outliers", "variable": "Gr Liv Area",
    "problema": f"Viviendas con superficie habitable > {UMBRAL_SUPERFICIE} sqft",
    "metodo": "Eliminacion por criterio de dominio",
    "filas_afectadas": eliminadas,
    "justificacion": "De Cock (2011) las documenta como atipicas; incluyen ventas parciales cuyo precio no refleja el valor de mercado",
})
registro.append({
    "bloque": "Outliers", "variable": "resto de variables continuas",
    "problema": "Valores extremos detectados por IQR y Z-score",
    "metodo": "Conservados sin modificacion",
    "filas_afectadas": 0,
    "justificacion": "Corresponden a variacion real del mercado inmobiliario; la asimetria se atendera con el escalador en Fase IV",
})

print(f"\nFilas eliminadas: {eliminadas} ({eliminadas/n_prev*100:.2f}% de la muestra)")
print(f"Forma actual: {df.shape[0]:,} × {df.shape[1]}")

,Order,Gr Liv Area,SalePrice,Sale Condition,Overall Qual,precio_por_sqft
1498,1499,5642,160000,Partial,10,28.36
1760,1761,4476,745000,Abnorml,10,166.44
1767,1768,4316,755000,Normal,10,174.93
2180,2181,5095,183850,Partial,10,36.08
2181,2182,4676,184750,Partial,10,39.51


Precio medio por sqft del resto del dataset: 120.23

Filas eliminadas: 5 (0.17% de la muestra)
Forma actual: 2,925 × 82


---
## 8. Reporte complementario: variables de baja varianza

**No se eliminan.** Se documentan para que la Fase IV decida con criterio propio:
una variable donde el 99.9% de los casos toma el mismo valor no aporta capacidad
discriminante, pero esa decisión pertenece a la selección de variables, no a la limpieza.

In [13]:
baja_varianza = []
for col in df.columns:
    vc = df[col].value_counts(normalize=True)
    if len(vc) and vc.iloc[0] > 0.95:
        baja_varianza.append({
            "variable": col,
            "valor_dominante": str(vc.index[0]),
            "concentracion_%": round(vc.iloc[0] * 100, 2),
            "categorias_distintas": int(df[col].nunique()),
        })

reporte_bv = pd.DataFrame(baja_varianza).sort_values("concentracion_%", ascending=False)
display(reporte_bv)
print(f"{len(reporte_bv)} variables con mas del 95% de concentracion en un solo valor.")
print("Entregado como insumo a Fase IV — sin accion en esta fase.")

,variable,valor_dominante,concentracion_%,categorias_distintas
1,Utilities,AllPub,99.90,3
9,Pool Area,0,99.62,12
10,Pool QC,Ninguno,99.62,5
0,Street,Pave,99.59,2
3,Condition 2,Norm,99.01,8
8,3Ssn Porch,0,98.74,31
6,Low Qual Fin SF,0,98.63,36
4,Roof Matl,CompShg,98.60,7
5,Heating,GasA,98.46,6
12,Misc Val,0,96.51,37


13 variables con mas del 95% de concentracion en un solo valor.
Entregado como insumo a Fase IV — sin accion en esta fase.


---
## 9. Validación final

Controles automáticos: si alguno falla, el notebook se detiene y el archivo
no se exporta.

In [14]:
errores = []

if df.isna().sum().sum() != 0:
    errores.append(f"Quedan {df.isna().sum().sum()} valores nulos")
if df.duplicated().any():
    errores.append("Existen filas duplicadas")
if df["PID"].duplicated().any():
    errores.append("PID no es unico")
if not df["Mo Sold"].between(1, 12).all():
    errores.append("Mo Sold fuera del rango 1-12")
if (df["Garage Yr Blt"] > df["Yr Sold"].max()).any():
    errores.append("Persisten anos de construccion imposibles")
if (df["SalePrice"] <= 0).any():
    errores.append("Precios de venta no positivos")
if df.shape[1] != df_raw.shape[1]:
    errores.append(f"Cambio el numero de columnas: {df_raw.shape[1]} -> {df.shape[1]}")

assert not errores, "VALIDACION FALLIDA:\n" + "\n".join(f"  - {e}" for e in errores)

print("VALIDACION SUPERADA")
print(f"  Filas    : {df_raw.shape[0]:,} -> {df.shape[0]:,}  ({df.shape[0]-df_raw.shape[0]:+d})")
print(f"  Columnas : {df_raw.shape[1]} -> {df.shape[1]}  (sin cambios)")
print(f"  Nulos    : {df_raw.isna().sum().sum():,} -> {df.isna().sum().sum()}")

VALIDACION SUPERADA
  Filas    : 2,930 -> 2,925  (-5)
  Columnas : 82 -> 82  (sin cambios)
  Nulos    : 15,749 -> 0


---
## 10. Comparativa pre/post y exportación

Insumo directo para la Fase V (auditoría) y la sección de resultados del artículo.

In [15]:
metricas_despues = resumen_numerico(df, "despues")
conteo_despues   = contadores(df, "despues")

comparativa = pd.DataFrame([conteo_antes, conteo_despues]).set_index("etapa").T
display(comparativa)

# Variación por variable numérica
piv = (pd.concat([metricas_antes, metricas_despues])
         .pivot(index="variable", columns="etapa",
                values=["media", "desv_std", "skewness", "curtosis", "nulos"]))
variacion = pd.DataFrame({
    "media_antes":    piv[("media", "antes")].round(2),
    "media_despues":  piv[("media", "despues")].round(2),
    "var_media_%":    ((piv[("media", "despues")] - piv[("media", "antes")])
                       / piv[("media", "antes")].replace(0, np.nan) * 100).round(2),
    "skew_antes":     piv[("skewness", "antes")].round(3),
    "skew_despues":   piv[("skewness", "despues")].round(3),
    "curtosis_antes": piv[("curtosis", "antes")].round(3),
    "curtosis_despues": piv[("curtosis", "despues")].round(3),
    "nulos_antes":    piv[("nulos", "antes")].astype(int),
    "nulos_despues":  piv[("nulos", "despues")].astype(int),
})
display(variacion.sort_values("nulos_antes", ascending=False).head(15))

etapa,antes,despues
filas,2930.000,2925.0
columnas,82.000,82.0
celdas_nulas,15749.000,0.0
pct_celdas_nulas,6.555,0.0
columnas_con_nulos,27.000,0.0
duplicados_exactos,0.000,0.0
variables_numericas,39.000,39.0
variables_no_numericas,43.000,43.0


,media_antes,media_despues,var_media_%,skew_antes,skew_despues,curtosis_antes,curtosis_despues,nulos_antes,nulos_despues
variable,,,,,,,,,
Lot Frontage,69.22,69.30,0.11,1.499,1.088,11.235,8.565,490,0
Garage Yr Blt,1978.13,1976.17,-0.10,-0.385,-0.693,1.827,-0.260,159,0
Mas Vnr Area,101.90,99.92,-1.94,2.607,2.578,9.287,9.151,23,0
Bsmt Half Bath,0.06,0.06,-1.02,3.941,3.968,14.922,15.159,2,0
Bsmt Full Bath,0.43,0.43,-0.29,0.617,0.617,-0.748,-0.755,2,0
BsmtFin SF 1,442.63,437.95,-1.06,1.416,0.822,6.859,0.087,1,0
Bsmt Unf SF,559.26,558.76,-0.09,0.923,0.925,0.410,0.413,1,0
Garage Cars,1.77,1.76,-0.13,-0.220,-0.221,0.245,0.250,1,0
BsmtFin SF 2,49.72,49.79,0.14,4.140,4.137,18.781,18.750,1,0


In [16]:
bitacora = pd.DataFrame(registro)[
    ["bloque", "variable", "problema", "metodo", "filas_afectadas", "justificacion"]
]

# Exportación
df.to_csv(PROCESSED, index=False)
pd.concat([metricas_antes, metricas_despues]).to_csv(DOCS / "metricas_pre_post.csv", index=False)
variacion.reset_index().to_csv(DOCS / "variacion_por_variable.csv", index=False)
comparativa.reset_index().to_csv(DOCS / "comparativa_global.csv", index=False)
bitacora.to_csv(DOCS / "bitacora_operaciones.csv", index=False)
reporte_outliers.to_csv(DOCS / "reporte_outliers.csv", index=False)
reporte_bv.to_csv(DOCS / "reporte_baja_varianza.csv", index=False)
nulos.reset_index().rename(columns={"index": "variable"}).to_csv(DOCS / "reporte_nulos_inicial.csv", index=False)

print("Archivos generados:")
print(f"  {PROCESSED.relative_to(ROOT)}  ({df.shape[0]:,} x {df.shape[1]})")
for f in sorted(DOCS.glob("*.csv")):
    print(f"  docs/{f.name}")

print(f"\nOperaciones registradas en la bitacora: {len(bitacora)}")
print(f"Total de celdas intervenidas: {int(bitacora['filas_afectadas'].sum()):,}")
display(bitacora)

Archivos generados:
  data\processed\AmesHousing.csv  (2,925 x 82)
  docs/bitacora_operaciones.csv
  docs/comparativa_global.csv
  docs/metricas_pre_post.csv
  docs/reporte_baja_varianza.csv
  docs/reporte_nulos_inicial.csv
  docs/reporte_outliers.csv
  docs/variacion_por_variable.csv

Operaciones registradas en la bitacora: 33
Total de celdas intervenidas: 18,270


,bloque,variable,problema,metodo,filas_afectadas,justificacion
0,Nulos,Garage Type,Tipo de garaje declarado sin superficie ni cap...,Reclasificado como ausencia de garaje,1,Inconsistencia interna: un garaje sin area ni ...
1,Nulos,Pool QC,NaN por ausencia del atributo (no dato perdido),"Codificado como categoria ""Ninguno""",2917,Verificado contra variable testigo: coincidenc...
2,Nulos,Misc Feature,NaN por ausencia del atributo (no dato perdido),"Codificado como categoria ""Ninguno""",2824,Verificado contra variable testigo: coincidenc...
3,Nulos,Alley,NaN por ausencia del atributo (no dato perdido),"Codificado como categoria ""Ninguno""",2732,Verificado contra variable testigo: coincidenc...
4,Nulos,Fence,NaN por ausencia del atributo (no dato perdido),"Codificado como categoria ""Ninguno""",2358,Verificado contra variable testigo: coincidenc...
5,Nulos,Fireplace Qu,NaN por ausencia del atributo (no dato perdido),"Codificado como categoria ""Ninguno""",1422,Verificado contra variable testigo: coincidenc...
6,Nulos,Garage Type,NaN por ausencia del atributo (no dato perdido),"Codificado como categoria ""Ninguno""",158,Verificado contra variable testigo: coincidenc...
7,Nulos,Garage Finish,NaN por ausencia del atributo (no dato perdido),"Codificado como categoria ""Ninguno""",159,Verificado contra variable testigo: coincidenc...
8,Nulos,Garage Qual,NaN por ausencia del atributo (no dato perdido),"Codificado como categoria ""Ninguno""",159,Verificado contra variable testigo: coincidenc...
9,Nulos,Garage Cond,NaN por ausencia del atributo (no dato perdido),"Codificado como categoria ""Ninguno""",159,Verificado contra variable testigo: coincidenc...


---
## 11. Verificación de ida y vuelta

El archivo exportado se vuelve a leer desde disco para confirmar que conserva
exactamente lo que se escribió. Esta comprobación existe porque una etiqueta mal
elegida puede reintroducir nulos al leer el CSV sin que nada falle visiblemente.

In [17]:
df_check = pd.read_csv(PROCESSED)

pruebas = {
    "Forma identica":            df_check.shape == df.shape,
    "Sin nulos tras la lectura": df_check.isna().sum().sum() == 0,
    "Mismas columnas":           list(df_check.columns) == list(df.columns),
    "Categoria de ausencia conservada":
        (df_check["Pool QC"] == ETIQUETA_AUSENCIA).sum() == (df["Pool QC"] == ETIQUETA_AUSENCIA).sum(),
}

for nombre, ok in pruebas.items():
    print(f"  [{'OK' if ok else 'FALLA'}] {nombre}")

assert all(pruebas.values()), "El archivo exportado no conserva el contenido"
print(f"\nArchivo verificado: {df_check.shape[0]:,} x {df_check.shape[1]}, 0 nulos.")
print('Etiqueta de ausencia:', ETIQUETA_AUSENCIA,
      f'({int((df_check == ETIQUETA_AUSENCIA).sum().sum()):,} celdas)')

  [OK] Forma identica
  [OK] Sin nulos tras la lectura
  [OK] Mismas columnas
  [OK] Categoria de ausencia conservada

Archivo verificado: 2,925 x 82, 0 nulos.
Etiqueta de ausencia: Ninguno (15,050 celdas)


---
## Entregable de la Fase III

| Archivo | Contenido | Destinatario |
|---|---|---|
| `data/processed/AmesHousing.csv` | Dataset limpio | Persona 4 (Transformación) |
| `docs/metricas_pre_post.csv` | Descriptivos antes/después | Persona 5 (Auditoría) |
| `docs/variacion_por_variable.csv` | Δ media, skewness, curtosis | Persona 5 / artículo |
| `docs/comparativa_global.csv` | Indicadores globales | Artículo, sección resultados |
| `docs/bitacora_operaciones.csv` | Registro de cada decisión | Rigor metodológico (35%) |
| `docs/reporte_outliers.csv` | IQR vs Z-score por variable | Artículo, discusión |
| `docs/reporte_baja_varianza.csv` | Variables casi constantes | Persona 4 |

**Pendiente para Fase IV:** escalamiento (la asimetría residual justifica evaluar
`RobustScaler` frente a `StandardScaler`), encoding de las 43 variables categóricas
—con atención a las ordinales de calidad, que admiten `OrdinalEncoder`— y creación
de variables derivadas.